# GC002 公平架构比较：Phase 0/1 工程准备
仅审计历史元数据、验证 KV 正确性、搜索参数匹配候选；不加载历史模型 Checkpoint、不训练、不扩大 Token 预算。
开启 Internet 和原有 GITHUB_TOKEN、KAGGLE_API_TOKEN。可使用 CPU；如果 Notebook 已启用 GPU，则额外测量随机权重的工程推理性能，仍不训练。点击 Save & Run All 自动发布准备报告。
CPU 检查通过不代表 GPU 性能或正式实验就绪。Phase 2 未冻结，Phase 3/4 禁止训练。总预算保持两小时。


In [ ]:
from pathlib import Path
import json, os, signal, subprocess, sys, time

BRANCH = 'codex/epclm-gc002-fair-architecture'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-epclm-core-001'
HARD_LIMIT_SECONDS = 7200
deadline = time.monotonic() + HARD_LIMIT_SECONDS

if not os.environ.get('KAGGLE_API_TOKEN', '').strip():
    from kaggle_secrets import UserSecretsClient
    os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN').strip()

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix':
            os.killpg(process.pid, signal.SIGTERM)
        else:
            process.terminate()
        try:
            output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix':
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Hard two-hour limit reached. Kaggle Output remains under ' + '/kaggle/working/epclm-gc002-preparation')
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Kaggle Output remains under ' + '/kaggle/working/epclm-gc002-preparation')

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'reset', '--hard', 'FETCH_HEAD'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH, REPOSITORY, str(REPO)])

run_bounded([sys.executable, str(REPO / "research/projects/cellular-language-models/experiments/epclm-core-001/gc002-fair-architecture/run.py"), "--prepare-only", "--output", "/kaggle/working/epclm-gc002-preparation", "--device", ("cuda" if __import__("torch").cuda.is_available() else "cpu"), "--recover-input-assets", "--publish"], cwd=REPO, timeout_seconds=6000)
